In [ ]:
!git clone https://github.com/toannn20/LINEA.git

Cloning into 'LINEA'...
remote: Enumerating objects: 258, done.
remote: Counting objects: 100% (258/258), done.
remote: Compressing objects: 100% (139/139), done.
remote: Total 258 (delta 125), reused 239 (delta 112), pack-reused 0 (from 0)
Receiving objects: 100% (258/258), 3.05 MiB | 8.43 MiB/s, done.
Resolving deltas: 100% (125/125), done.


In [ ]:
%cd /content/LINEA
!pip install -r requirements.txt
!unzip /content/drive/MyDrive/Datasets/line_detect/data.zip -d ./data

/content/LINEA
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.8/46.8 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.2/87.2 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.2/256.2 kB 23.7 MB/s eta 0:00:00
Archive:  /content/drive/MyDrive/Datasets/line_detect/data.zip
   creating: ./data/annotations/
  inflating: ./data/annotations/lines_train_ann.json  
  inflating: ./data/annotations/lines_val_ann.json  
  inflating: ./data/annotations/lines_test_ann.json  
   creating: ./data/test/
  inflating: ./data/test/gg_architecture_094ffe9a22fb863211de7386157fd9da6558a86e.jpg  
  inflating: ./data/test/gg_architecture_0f4c1ea3d29177bf2433aa7076331568f309cdd3.jpg  
  inflating: ./data/test/gg_architecture_1d46c1729120d28c61962c43b7244351f0694210.jpg  
  inflating: ./data/test/gg_architecture_229eeca76791ed22d61c7a7fd87f5774de188942.jpg  
  inflating: ./data/test/gg_architecture_2c35f23eaf15531f689958452d77eb07a7aac797.jpg  
  inflating: ./data/t

In [ ]:
!CUDA_VISIBLE_DEVICES=0 torchrun --master_port=7777 --nproc_per_node=1 main.py -c configs/linea/linea_hgnetv2_n.py --coco_path /content/LINEA/data --amp --options epochs=5

/usr/local/lib/python3.12/dist-packages/torch/distributed/c10d_logger.py:83: UserWarning: barrier(): using the device under current context. You can specify `device_id` in `init_process_group` to mute this warning.
  return func(*args, **kwargs)
[rank0]:[W226 16:54:04.412281643 ProcessGroupNCCL.cpp:5138] Guessing device ID based on global rank. This can cause a hang if rank to GPU mapping is heterogeneous. You can specify device_id in init_process_group()
Initialized distributed mode...
Namespace(config_file='configs/linea/linea_hgnetv2_n.py', options={'epochs': 5}, coco_path='/content/LINEA/data', device='cuda', seed=42, resume='', start_epoch=0, eval=False, num_workers=10, find_unused_params=False, world_size=1, rank=0, local_rank=None, amp=True, gpu=0, distributed=True, data_aug_scales=[(640, 640)], data_aug_max_size=1333, data_aug_scales2_resize=[400, 500, 600], data_aug_scales2_crop=[384, 600], data_aug_scale_overlap=None, batch_size_train=8, batch_size_val=64, lr=0.0008, weight_d

In [ ]:
!python tools/inference/torch_inf.py -c configs/linea/linea_hgnetv2_n.py -r /content/LINEA/output/linea_hgnetv2_n/checkpoint0004.pth --input /content/LINEA/data/test/val/gg_architecture_094ffe9a22fb863211de7386157fd9da6558a86e.jpg --device cuda:0

Image processing complete. Result saved as 'torch_results.jpg'.
Image processing complete.


In [ ]:
!python tools/visualization/line_attention.py -c ./configs/linea/linea_hgnetv2_n.py --resume /content/LINEA/output/linea_hgnetv2_n/checkpoint0004.pth --data-path /content/LINEA/data/test -d cuda --num_images 10

building val_dataloader with batch_size=1...
loading annotations into memory...
Done (t=0.01s)
creating index...
index created!
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to a

In [ ]:
!python tools/visualization/backbone_encoder.py -c ./configs/linea/linea_hgnetv2_n.py --resume /content/LINEA/output/linea_hgnetv2_n/checkpoint0004.pth --data-path /content/LINEA/data/test -d cuda --num_images 10

building val_dataloader with batch_size=1...
loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to a

In [ ]:
import torch
checkpoint = torch.load('/content/LINEA/output/linea_hgnetv2_n/checkpoint0004.pth', weights_only=False)
for k, v in checkpoint.items():
    print(k, v.shape if hasattr(v, 'shape') else type(v))

model <class 'collections.OrderedDict'>
optimizer <class 'dict'>
lr_scheduler <class 'dict'>
warmup_scheduler <class 'NoneType'>
epoch <class 'int'>
args <class 'argparse.Namespace'>


In [ ]:
torch.save(checkpoint['model'], '/content/linea_ghmetv2_n_final.pth')